# Manim Animation Studio

Kaggle notebook for pasting Manim code, rendering videos, previewing, and downloading MP4.

Run all cells from top to bottom. Open the public Gradio link printed by the final cell.

## 1. Check environment

In [1]:
import os
import sys
import shutil

print("Python:", sys.version.split()[0])
print("Working directory:", os.getcwd())
print("FFmpeg:", shutil.which("ffmpeg"))
print("Disk free (GB):", round(shutil.disk_usage("/kaggle/working").free / (1024 ** 3), 2))

Python: 3.13.15
Working directory: /kaggle/working
FFmpeg: /usr/bin/ffmpeg
Disk free (GB): 19.5


## 2. Install system dependencies

In [2]:
!apt-get update -qq
!apt-get install -y -qq build-essential pkg-config libcairo2-dev libpango1.0-dev ffmpeg texlive-latex-base texlive-latex-extra texlive-fonts-recommended

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Preconfiguring packages ...
Selecting previously unselected package fonts-lato.
(Reading database ... 129796 files and directories currently installed.)
Preparing to unpack .../00-fonts-lato_2.015-1_all.deb ...
Unpacking fonts-lato (2.015-1) ...
Selecting previously unselected package at-spi2-common.
Preparing to unpack .../01-at-spi2-common_2.52.0-1build1_all.deb ...
Unpacking at-spi2-common (2.52.0-1build1) ...
Selecting previously unselected package libatspi2.0-0t64:amd64.
Preparing to unpack .../02-libatspi2.0-0t64_2.52.0-1build1_amd64.deb ...
Unpa

## 3. Install Python packages

In [3]:
%pip install -q --upgrade pip wheel
%pip install -q manim gradio
%pip install -q "setuptools<82" 

Note: you may need to restart the kernel to use updated packages.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## 4. Load UI dependencies

In [4]:
import ast
import subprocess
import tempfile
import shutil
from pathlib import Path
import gradio as gr

WORK_DIR = Path("/kaggle/working/manim_studio")
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("Manim:", subprocess.run(["manim", "--version"], capture_output=True, text=True).stdout.strip())
print("Gradio:", gr.__version__)

Manim: Manim Community v0.22.0
Gradio: 6.26.0


## 5. Launch Animation Studio

In [23]:

# ── Configuration ──────────────────────────────────────────────────────

RATIOS = {
    "16:9 — YouTube": (1920, 1080),
    "9:16 — Shorts / Reels": (1080, 1920),
    "1:1 — Square": (1080, 1080),
    "4:3 — Classic": (1440, 1080),
}

WORK_DIR = os.path.join(os.getcwd(), "renders")
os.makedirs(WORK_DIR, exist_ok=True)

MANIM_PROMPT = r"""
ROLE
You are a senior Manim Community Edition (CE) engineer, scientific visual
storyteller, motion designer, and careful Python reviewer. Create polished
educational animations that are visually clear, calm, accurate, and reliable.

USER INPUT
The user may provide a topic, narration script, timestamps, visual instructions,
equations, examples, or reference notes.

TOPIC / SCRIPT / VISUAL DIRECTION:
[PASTE THE USER'S TOPIC, SCRIPT, TIMESTAMPS, OR VISUAL INSTRUCTIONS HERE]

SELECTED ASPECT RATIO: [ASPECT RATIO]

PRIMARY GOAL
Return one complete, self-contained Manim CE Python program that explains the
provided content with smooth pacing, purposeful visuals, medium-sized readable
text, balanced layouts, and no unintended overlap or clipping.

Never claim that code was executed, rendered, or verified unless it actually was.
Prefer simple, supported Manim CE techniques over fragile or version-sensitive tricks.

==================================================
1. UNDERSTAND THE SCRIPT AND PLAN THE TIMELINE
==================================================
- Read the complete user input before writing code.
- Identify the learning goal, key concepts, examples, transitions, and conclusion.
- Plan the scenes and timeline before implementing the animation.
- If timestamps are supplied, treat them as the intended target timeline.
- Preserve the timestamp order and match each visual to its correct script segment.
- Account for every animation run_time, transition, and wait in the timeline.
- Avoid cumulative timing drift and do not show a concept before its script segment.
- If timestamps are absent, choose pacing based on the content and complexity.
- Aim for 60 seconds or more only when the content supports a meaningful explanation.
- Never pad a short script with empty waits just to reach a duration target.
- Keep the animation calm and readable rather than rushed or overloaded.
- As a general guide, allow around 2–5 seconds for a simple visual idea and longer
  for equations, multi-step diagrams, and important conclusions.
- When a supplied segment is short, simplify the visual rather than cramming it.
- Text timestamps are not proof of audio synchronization. Do not claim verified
  audio sync unless actual audio timing has been measured.
- Keep a timeline plan in code comments only if useful; final output must still
  contain complete runnable Python code.

==================================================
2. PREMIUM VISUAL IDENTITY
==================================================
BACKGROUND AND COMPOSITION
- Use a pure black or near-black background consistently.
- Keep each composition minimal, cinematic, intentional, and easy to understand.
- Show one main visual focus at a time.
- Use particles, glow, grids, and decorative details only when they explain the topic.
- Avoid clutter, excessive glow, constant background motion, unnecessary icons,
  decorative borders, and visual effects that distract from the lesson.
- Do not add channel logos, channel names, watermarks, signatures, or branding
  unless the user explicitly asks for them.

COLOR SYSTEM
- Main text: white or soft white.
- Supporting text: readable light gray.
- Main diagram accents: cyan or electric blue.
- Yellow: key terms, important values, or focused highlights.
- Red: errors, danger, incorrect states, or negative outcomes.
- Green: correct results, success, or positive outcomes.
- Violet is optional and should be used sparingly.
- Keep color meanings consistent throughout the animation.
- Do not color every word differently.
- Maintain high contrast against the dark background.
- Never rely on red/green alone to communicate meaning; add a label, shape, or icon.

==================================================
3. PROFESSIONAL TYPOGRAPHY AND CONSISTENT SIZES
==================================================
- Prefer Space Grotesk or Exo 2 for modern headings and body text when installed.
- Use Orbitron or Michroma only for short technical titles.
- Use Fira Code or JetBrains Mono for code, terminal output, and technical labels.
- Audiowide may be used sparingly for a short accent, never for paragraphs.
- Always provide a reliable fallback such as DejaVu Sans if the preferred font
  is unavailable. Never assume a custom font or external asset exists.
- Establish a consistent type scale and reuse it across all scenes.
- Titles should be moderately sized, not oversized.
- Body text should be comfortable to read without dominating the frame.
- Captions must use a consistent size across scenes.
- Supporting labels can be smaller, but must remain legible.
- Do not randomly resize equivalent titles, captions, labels, or body text.
- Avoid tiny text, giant headings, excessive all-caps, and decorative body fonts.
- Use bold only when it provides meaningful emphasis.
- For text size, consider the final output resolution and the selected aspect ratio.

==================================================
4. STRICT TEXT FITTING, BOXES, AND SAFE AREAS
==================================================
Text must not overlap, overflow a box, or cross the video frame.

BEFORE POSITIONING TEXT
- Consider the actual rendered width and height of each text object.
- Check both width and height for multi-line text.
- Use consistent safe margins on all sides of the frame.
- Recheck layout after scaling, transforming, moving the camera, or changing ratio.
- Prefer measured dimensions over assumptions based on character count.
- If text does not fit, shorten it, wrap it, or split it across sequential scenes.
- Never shrink text so much that it becomes unreadable.

TEXT INSIDE BOXES OR CARDS
- Size the box around the final text when practical; otherwise fit the text to the box.
- Leave visible internal padding on every side.
- Ensure the entire text bounding box stays inside the card's inner boundaries.
- Keep neighbouring cards separated by a clear gap.
- Never put a long sentence or paragraph inside a small box.
- Keep text away from borders, arrows, connectors, icons, and other labels.
- Do not let arrows or lines pass through text.
- If content is too dense, use multiple cards or scenes instead of shrinking everything.
- After any transform or scale, recheck the text and box positions.

CAPTIONS AND SUBTITLES
- Use a consistent caption size and a defined caption safe area.
- Keep captions within the left and right frame margins and away from the bottom edge.
- Do not cover equations, diagrams, labels, or the main subject.
- Keep captions concise and visible long enough to read.
- For vertical video, avoid common social-video interface areas.
- Do not place multiple captions on top of one another.
- Verify that the final frame contains no clipped or partially hidden text.

LAYOUT SAFETY
- Keep titles, diagrams, captions, and supporting labels in distinct visual areas.
- Place labels relative to the object they describe.
- Remove, fade, or transform old objects intentionally before reusing the same area.
- Do not leave stale updaters or invisible objects affecting later scenes.
- Before returning code, review each scene for collisions and frame overflow.

==================================================
5. SLOW, SMOOTH, PURPOSEFUL ANIMATION
==================================================
- Motion should feel deliberate, smooth, and premium—not rushed.
- Use moderate animation durations and give viewers time to understand each change.
- Prefer appropriate built-in animations such as FadeIn, FadeOut, Create, Write,
  GrowArrow, Transform, ReplacementTransform, and LaggedStart.
- Prefer commonly supported rate functions such as smooth and linear.
- Never reference an undefined easing function such as ease_in_out_sine unless
  you explicitly define a valid implementation.
- Do not invent animation classes or transformations such as ScaleTransform.
  Use a supported alternative such as Transform when appropriate.
- Avoid many tiny animations played back-to-back, excessive bouncing, sudden jumps,
  fast zooms, spinning without purpose, and constant movement behind text.
- Do not start an animation before its target object exists.
- Ensure each animation finishes before later code relies on its final state.
- Use waits only when they aid comprehension or match supplied timing.
- Keep camera movement restrained and stable.
- Use 3D only when it improves understanding; preserve label readability.
- Avoid unnecessary updaters. If used, ensure they reference valid objects and do
  not create unmanaged objects on every frame.
- Keep movement paths and final positions inside the frame.
- Reveal equations, diagrams, and results at a pace that viewers can follow.
- Ensure the last scene resolves cleanly and the final frame remains readable.

==================================================
6. EDUCATIONAL STORYTELLING AND ACCURACY
==================================================
- Every visual must explain a specific concept from the script.
- Preserve scientific, mathematical, and technical accuracy.
- Use diagrams, step-by-step transformations, graphs, equations, comparisons,
  timelines, labeled components, and analogies when they improve understanding.
- Reveal complex systems progressively instead of showing everything at once.
- Explain cause and effect visually when possible.
- Avoid decorative motion that teaches nothing.
- Do not invent facts, equations, values, or conclusions without a clear reason.
- If a concept is complex, simplify the visual without making it misleading.
- Do not omit important parts of the user's script.
- End with a clear, visually balanced takeaway appropriate to the content.

==================================================
7. ASPECT RATIO AND RESPONSIVE LAYOUT
==================================================
SELECTED ASPECT RATIO: [ASPECT RATIO]

- Compose specifically for the selected ratio: 16:9, 9:16, 1:1, or 4:3.
- Do not design only for widescreen and then crop it for vertical video.
- Keep all essential content inside safe margins.
- For vertical video, stack elements and use fewer words per line.
- For widescreen, use horizontal space without stretching text across the frame.
- For square video, simplify and stack elements when necessary.
- Recalculate text widths, box dimensions, caption positions, and diagram layout
  for the selected ratio.
- Do not assume fixed coordinates work equally well in every aspect ratio.
- Ensure labels remain readable and avoid collisions in all selected formats.

==================================================
8. MANIM CE AND PYTHON COMPATIBILITY
==================================================
- Return complete runnable Python code, not fragments or pseudocode.
- Include all imports, constants, helper functions, and Scene/ThreeDScene classes.
- Use standard Manim Community Edition APIs, not ManimGL-only APIs.
- Do not invent unsupported keyword arguments for Arrow, Text, MathTex, Line,
  Sphere, camera methods, or animation classes.
- Prefer conservative, widely supported methods over obscure conveniences.
- Do not use ScaleTransform or any other name unless it is actually defined and
  supported. Prefer Transform or ReplacementTransform when suitable.
- Never reference undefined functions, variables, colors, classes, or rate functions.
- Define helper functions before first use.
- Use compatible Manim color objects. Do not pass raw strings to functions that
  expect Manim color objects.
- For color interpolation, use supported functions and compatible values. If unsure,
  use discrete color choices instead of interpolation.
- Avoid dependencies on external files, internet downloads, optional packages,
  or unavailable fonts. Generate visuals procedurally unless assets are supplied.
- Use valid NumPy array dimensions and 3D coordinates when NumPy is needed.
- Ensure VGroups and animation targets contain the intended objects.
- Avoid stale references after Transform or ReplacementTransform.
- Ensure every object is created before it is animated.
- Use valid scene class names ending in Scene or ThreeDScene.
- Prefer one main scene unless multiple scenes are necessary.
- Do not silently rely on a class or method from a different Manim version.
- If uncertain about an API, choose a simpler, well-established Manim CE approach.

==================================================
9. FINAL STATIC REVIEW CHECKLIST
==================================================
Before returning code, review it systematically:

A. Syntax and structure
- Imports, quotes, brackets, indentation, and class definitions are correct.
- Every referenced name is defined or imported.
- Each scene has a valid construct method.
- No Markdown fence markers appear inside the Python source.

B. API compatibility
- All class names, methods, arguments, colors, and rate functions are valid Manim CE.
- No unsupported constructor arguments or undefined transformations are present.
- No external dependency is silently assumed.

C. Layout
- Every object stays inside the frame.
- Text fits inside boxes with padding.
- Captions, titles, diagrams, and labels do not overlap.
- The layout matches the selected aspect ratio.
- Font sizes are consistent and readable.

D. Animation and timeline
- Objects exist before they are animated.
- Old objects and transformations are managed intentionally.
- Animation durations and waits respect supplied timestamps.
- Motion is smooth, calm, and readable rather than excessively fast.
- Updaters and trackers reference valid objects.
- The final frame is complete and readable.

E. Completeness and honesty
- The full script is represented in chronological order.
- Complex effects are simplified rather than implemented with fragile code.
- Do not claim execution or render verification unless actually performed.
- If a requirement cannot be met reliably, choose the most robust alternative.

==================================================
10. FINAL OUTPUT FORMAT
==================================================
Return ONLY the complete Manim CE Python code inside one Python code block.
Do not include explanations before or after the code.
Do not return partial snippets, TODOs, unimplemented placeholders, or pseudocode.
"""
# ── Premium dark dashboard styling ─────────────────────────────────────

APP_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&family=JetBrains+Mono:wght@400;500&display=swap');

:root {
    --bg: #07090F;
    --bg-2: #0A0D15;
    --surface: rgba(20, 26, 40, 0.72);
    --surface-solid: #111723;
    --surface2: #171F2E;
    --border: rgba(120, 140, 180, 0.16);
    --border-strong: rgba(120, 140, 180, 0.28);
    --text: #F3F6FC;
    --text-2: #C3CCDD;
    --muted: #8593AB;
    --indigo: #6366F1;
    --blue: #4F7CFF;
    --cyan: #22D3EE;
    --violet: #8B5CF6;
    --green: #34D399;
    --grad: linear-gradient(110deg, #5B6CFF 0%, #3B9BFF 55%, #22D3EE 100%);
    --radius: 14px;
}

* { box-sizing: border-box; }

/* ── Full-window scroll with visible scrollbar ── */
html, body {
    margin: 0 !important;
    padding: 0 !important;
    height: auto !important;
    min-height: 100% !important;
    background: var(--bg) !important;
    color: var(--text) !important;
    overflow-x: hidden !important;
    overflow-y: scroll !important;
    scrollbar-width: thin;
    scrollbar-color: #3A4770 #0A0D15;
}

html::-webkit-scrollbar,
body::-webkit-scrollbar { width: 12px; }

html::-webkit-scrollbar-track,
body::-webkit-scrollbar-track {
    background: #0A0D15;
    border-left: 1px solid rgba(120, 140, 180, 0.12);
}

html::-webkit-scrollbar-thumb,
body::-webkit-scrollbar-thumb {
    background: linear-gradient(180deg, #5B6CFF, #22D3EE);
    border-radius: 10px;
    border: 3px solid #0A0D15;
}

html::-webkit-scrollbar-thumb:hover,
body::-webkit-scrollbar-thumb:hover { filter: brightness(1.15); }

.gradio-container {
    position: relative !important;
    width: 100% !important;
    max-width: 100% !important;
    height: auto !important;
    min-height: 100dvh !important;
    margin: 0 !important;
    padding: 0 !important;
    overflow: visible !important;
    color: var(--text) !important;
    font-family: 'Inter', -apple-system, BlinkMacSystemFont, 'Segoe UI', Arial, sans-serif !important;
    font-size: 13px !important;
    -webkit-font-smoothing: antialiased;
    background:
        radial-gradient(900px 500px at 8% -8%, rgba(99,102,241,.18), transparent 60%),
        radial-gradient(800px 520px at 100% 0%, rgba(34,211,238,.11), transparent 60%),
        radial-gradient(700px 500px at 50% 120%, rgba(139,92,246,.10), transparent 60%),
        var(--bg) !important;
}

.gradio-container .main, .gradio-container .wrap, .gradio-container .contain {
    background: transparent !important;
}

/* ── Top bar (sticky) ── */
#topbar {
    position: sticky !important;
    top: 0 !important;
    z-index: 50 !important;
    height: 62px !important;
    display: flex !important;
    align-items: center !important;
    justify-content: space-between !important;
    padding: 0 26px !important;
    background: rgba(10, 13, 21, 0.85) !important;
    backdrop-filter: blur(14px) !important;
    -webkit-backdrop-filter: blur(14px) !important;
    border-bottom: 1px solid var(--border) !important;
}

#topbar::after {
    content: "";
    position: absolute;
    left: 0; right: 0; bottom: -1px;
    height: 1px;
    background: linear-gradient(90deg, transparent, rgba(99,102,241,.7), rgba(34,211,238,.7), transparent);
    opacity: .55;
}

#brand {
    display: flex !important;
    align-items: center !important;
    gap: 12px !important;
    color: var(--text) !important;
}

#brand .logo {
    width: 32px;
    height: 32px;
    flex: 0 0 32px;
    border-radius: 9px;
    display: inline-flex;
    align-items: center;
    justify-content: center;
    background: var(--grad);
    box-shadow: 0 6px 20px rgba(79,124,255,.38), inset 0 1px 0 rgba(255,255,255,.35);
    color: #fff;
    font-size: 15px;
    font-weight: 800;
}

#brand .title {
    display: flex;
    flex-direction: column;
    line-height: 1.15;
}

#brand .title b {
    font-size: 15px;
    font-weight: 700;
    letter-spacing: -0.2px;
    color: var(--text);
}

#brand .title small {
    margin-top: 3px;
    color: var(--muted) !important;
    font-size: 10.5px !important;
    font-weight: 500 !important;
    letter-spacing: .2px;
}

#topbar-badge {
    display: inline-flex !important;
    align-items: center !important;
    gap: 8px !important;
    color: #B6C8FF !important;
    background: rgba(99,102,241,.10) !important;
    border: 1px solid rgba(99,102,241,.30) !important;
    padding: 7px 13px !important;
    border-radius: 999px !important;
    font-size: 10px !important;
    font-weight: 700 !important;
    letter-spacing: 1px !important;
}

#topbar-badge .pulse {
    width: 7px; height: 7px;
    border-radius: 50%;
    background: var(--green);
    box-shadow: 0 0 0 0 rgba(52,211,153,.6);
    animation: pulse 2s infinite;
}

@keyframes pulse {
    0%   { box-shadow: 0 0 0 0 rgba(52,211,153,.55); }
    70%  { box-shadow: 0 0 0 7px rgba(52,211,153,0); }
    100% { box-shadow: 0 0 0 0 rgba(52,211,153,0); }
}

/* ── Layout ── */
#main-shell {
    display: grid !important;
    grid-template-columns: minmax(0, 1fr) minmax(0, 1fr) !important;
    gap: 18px !important;
    height: auto !important;
    min-height: calc(100dvh - 62px) !important;
    padding: 18px 22px 28px !important;
    overflow: visible !important;
    align-items: start !important;
    background: transparent !important;
    border: 0 !important;
}

.panel {
    min-width: 0 !important;
    min-height: 0 !important;
    height: auto !important;
    overflow: visible !important;
    display: flex !important;
    flex-direction: column !important;
    gap: 14px !important;
    padding: 0 !important;
    background: transparent !important;
    border: 0 !important;
}

/* ── Cards ── */
.card {
    background: var(--surface) !important;
    backdrop-filter: blur(10px) !important;
    -webkit-backdrop-filter: blur(10px) !important;
    border: 1px solid var(--border) !important;
    border-radius: var(--radius) !important;
    padding: 16px !important;
    gap: 10px !important;
    box-shadow: 0 10px 30px rgba(0,0,0,.28), inset 0 1px 0 rgba(255,255,255,.03) !important;
    transition: border-color .2s ease, box-shadow .2s ease !important;
    flex: 0 0 auto !important;
}

.card:hover {
    border-color: var(--border-strong) !important;
}

.section-label {
    display: flex !important;
    align-items: center !important;
    gap: 10px !important;
    color: var(--text) !important;
    font-size: 11.5px !important;
    font-weight: 700 !important;
    letter-spacing: 1px !important;
    text-transform: uppercase !important;
    margin: 0 !important;
}

.section-label .num {
    width: 22px;
    height: 22px;
    display: inline-flex;
    align-items: center;
    justify-content: center;
    border-radius: 7px;
    color: #fff;
    background: var(--grad);
    box-shadow: 0 4px 12px rgba(79,124,255,.35);
    font-size: 11px;
    font-weight: 700;
    letter-spacing: 0;
}

.hint {
    color: var(--muted) !important;
    font-size: 11.5px !important;
    line-height: 1.55 !important;
    margin: 0 0 2px !important;
}

/* ── Form elements ── */
.gradio-container label,
.gradio-container .block label span,
.gradio-container span[data-testid="block-info"] {
    color: var(--text-2) !important;
    font-size: 11px !important;
    font-weight: 600 !important;
    letter-spacing: .2px !important;
}

.gradio-container .block,
.gradio-container .form {
    background: transparent !important;
    border-color: transparent !important;
    box-shadow: none !important;
}

.gradio-container input,
.gradio-container textarea,
.gradio-container select {
    color: var(--text) !important;
    background: var(--surface2) !important;
    border: 1px solid var(--border) !important;
    border-radius: 10px !important;
    font-size: 12.5px !important;
    transition: border-color .15s ease, box-shadow .15s ease !important;
}

.gradio-container input:focus,
.gradio-container textarea:focus,
.gradio-container select:focus {
    border-color: rgba(99,102,241,.65) !important;
    box-shadow: 0 0 0 3px rgba(99,102,241,.16) !important;
    outline: none !important;
}

.gradio-container textarea::placeholder,
.gradio-container input::placeholder {
    color: #6F7E98 !important;
    opacity: 1 !important;
}

/* Code editors */
.gradio-container .cm-editor {
    background: #0C111B !important;
    color: #EAF0FC !important;
    border: 1px solid var(--border) !important;
    border-radius: 10px !important;
    font-family: 'JetBrains Mono', Consolas, monospace !important;
    font-size: 11.5px !important;
}

.gradio-container .cm-gutters {
    background: #090D15 !important;
    color: #56647E !important;
    border-right: 1px solid var(--border) !important;
}

.gradio-container .cm-activeLine,
.gradio-container .cm-activeLineGutter {
    background: rgba(99,102,241,.08) !important;
}

#code-input .cm-editor {
    min-height: 230px !important;
}

/* Accordion */
.gradio-container details {
    background: var(--surface2) !important;
    border: 1px solid var(--border) !important;
    border-radius: 11px !important;
    padding: 10px 12px !important;
    color: var(--text) !important;
}

.gradio-container summary {
    color: #C9D6FF !important;
    font-size: 12px !important;
    font-weight: 600 !important;
    cursor: pointer !important;
}

/* Buttons */
.gradio-container button {
    min-height: 34px !important;
    border-radius: 10px !important;
    color: #EAF0FC !important;
    font-size: 11.5px !important;
    font-weight: 600 !important;
    transition: filter .15s ease, transform .15s ease, box-shadow .15s ease !important;
}

#render-btn {
    width: 100% !important;
    min-height: 46px !important;
    margin-top: 6px !important;
    background: var(--grad) !important;
    color: #FFFFFF !important;
    border: 0 !important;
    border-radius: 12px !important;
    font-size: 13.5px !important;
    font-weight: 700 !important;
    letter-spacing: .3px !important;
    box-shadow: 0 8px 26px rgba(59,130,246,.34), inset 0 1px 0 rgba(255,255,255,.28) !important;
}

#render-btn:hover {
    filter: brightness(1.08);
    transform: translateY(-1px);
    box-shadow: 0 12px 32px rgba(59,130,246,.45), inset 0 1px 0 rgba(255,255,255,.3) !important;
}

#render-btn:active { transform: translateY(0); }

/* Video + file + log */
#video-preview {
    border-radius: 12px !important;
    overflow: hidden !important;
}

#video-preview video {
    display: block !important;
    width: 100% !important;
    max-height: 300px !important;
    object-fit: contain !important;
    background: #04060A !important;
    border: 1px solid var(--border) !important;
    border-radius: 12px !important;
}

#file-out {
    background: var(--surface2) !important;
    border: 1px dashed var(--border-strong) !important;
    border-radius: 12px !important;
    padding: 8px !important;
}

#status-box textarea {
    background: #060910 !important;
    color: #8FE8BE !important;
    border: 1px solid var(--border) !important;
    font-family: 'JetBrains Mono', Consolas, monospace !important;
    font-size: 10.5px !important;
    line-height: 1.55 !important;
}

footer { display: none !important; }

/* ── Responsive ── */
@media (max-width: 850px) {
    #main-shell {
        display: flex !important;
        flex-direction: column !important;
        padding: 14px !important;
    }

    #brand .title small,
    #topbar-badge {
        display: none !important;
    }
}
"""


# ── Render function ─────────────────────────────────────────────────────

def render_video(code, ratio, quality):
    if not code or not code.strip():
        raise gr.Error("Paste your complete Manim Python code first.")

    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        raise gr.Error(f"Syntax error on line {e.lineno}: {e.msg}")

    scenes = [
        node.name for node in tree.body
        if isinstance(node, ast.ClassDef)
        and any(
            (isinstance(base, ast.Name) and base.id.endswith("Scene"))
            or (isinstance(base, ast.Attribute) and base.attr.endswith("Scene"))
            for base in node.bases
        )
    ]

    if not scenes:
        raise gr.Error("No Scene or ThreeDScene class found in the code.")

    width, height = RATIOS[ratio]
    quality_flag = {
        "Draft — Fast": "-ql",
        "Preview — Balanced": "-qm",
        "Final — High quality": "-qh",
    }[quality]

    os.makedirs(WORK_DIR, exist_ok=True)
    job = Path(tempfile.mkdtemp(prefix="render_", dir=WORK_DIR))
    script = job / "animation.py"
    script.write_text(code, encoding="utf-8")

    last_video = None
    logs = []

    for scene in scenes:
        command = [
            "manim", quality_flag,
            "-r", f"{width},{height}",
            "--format=mp4", str(script), scene
        ]

        try:
            result = subprocess.run(
                command,
                cwd=str(job),
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                timeout=900
            )
        except subprocess.TimeoutExpired:
            raise gr.Error(
                "Render timed out. Try Draft quality or simplify the animation."
            )

        logs.append(f"SCENE: {scene}\n{result.stdout[-2500:]}")

        if result.returncode != 0:
            raise gr.Error("Render failed:\n" + result.stdout[-6000:])

        videos = list(job.rglob(f"{scene}.mp4"))
        if videos:
            last_video = max(videos, key=lambda p: p.stat().st_mtime)

    if not last_video or not last_video.exists():
        raise gr.Error("Rendered MP4 was not found.")

    output = Path(WORK_DIR) / f"{scenes[-1]}_{width}x{height}.mp4"
    shutil.copy2(last_video, output)

    status = (
        f"Render successful | {width} × {height} | {quality}\n"
        f"File: {output.name}\n\n"
        + "\n".join(logs)[-3000:]
    )

    return str(output), str(output), status


# ── Build the UI ────────────────────────────────────────────────────────

with gr.Blocks(title="Motionary") as app:

    gr.HTML("""
    <div id="topbar">
        <div id="brand">
            <span class="logo">M</span>
            <div class="title">
                <b>Motionary</b>
                <small>Cinematic educational video generator</small>
            </div>
        </div>
        <div id="topbar-badge"><span class="pulse"></span> RENDER ENGINE ONLINE</div>
    </div>
    """)

    with gr.Row(elem_id="main-shell"):

        # LEFT: prompt, code, settings
        with gr.Column(elem_classes=["panel"], scale=1):

            # Section 1 — Prompt
            with gr.Column(elem_classes=["card"]):
                gr.HTML("""
                <div class="section-label">
                    <span class="num">1</span> LLM Prompt
                </div>
                """)
                gr.HTML("""
                <div class="hint">
                    Copy this prompt into any LLM, add your topic or script,
                    then paste the generated Manim code below.
                </div>
                """)

                with gr.Accordion("📋  Copy This Prompt for Any LLM", open=False):
                    gr.Code(
                        value=MANIM_PROMPT,
                        language="markdown",
                        label="Pre-made animation prompt",
                        interactive=False,
                        lines=6
                    )

            # Section 2 — Code
            with gr.Column(elem_classes=["card"]):
                gr.HTML("""
                <div class="section-label">
                    <span class="num">2</span> Manim Python Code
                </div>
                """)
                gr.HTML("""
                <div class="hint">
                    Paste the complete Python code returned by your LLM.
                </div>
                """)

                code_input = gr.Code(
                    label="Python code",
                    language="python",
                    lines=8,
                    elem_id="code-input"
                )

            # Section 3 — Settings
            with gr.Column(elem_classes=["card"]):
                gr.HTML("""
                <div class="section-label">
                    <span class="num">3</span> Render Settings
                </div>
                """)

                with gr.Row():
                    ratio_input = gr.Dropdown(
                        choices=list(RATIOS.keys()),
                        value="16:9 — YouTube",
                        label="Aspect ratio"
                    )
                    quality_input = gr.Dropdown(
                        choices=[
                            "Draft — Fast",
                            "Preview — Balanced",
                            "Final — High quality"
                        ],
                        value="Draft — Fast",
                        label="Render quality"
                    )

                render_button = gr.Button(
                    "▶  Render Video",
                    variant="primary",
                    elem_id="render-btn"
                )

        # RIGHT: preview, download, logs
        with gr.Column(elem_classes=["panel"], scale=1):

            with gr.Column(elem_classes=["card"]):
                gr.HTML("""
                <div class="section-label">
                    <span class="num">4</span> Preview & Download
                </div>
                """)

                video_output = gr.Video(
                    label="Video preview",
                    elem_id="video-preview",
                    height=270
                )

                file_output = gr.File(
                    label="Download MP4",
                    elem_id="file-out"
                )

                status_output = gr.Textbox(
                    label="Render log",
                    lines=6,
                    interactive=False,
                    elem_id="status-box"
                )

            gr.HTML("""
            <div style="
                margin-top:auto;
                padding:12px 14px;
                background:linear-gradient(100deg, rgba(99,102,241,.10), rgba(34,211,238,.06));
                border:1px solid rgba(99,102,241,.22);
                border-radius:12px;
                color:#9BA8BD;
                font-size:11px;
                line-height:1.55;">
                <span style="color:#A5B8FF;font-weight:700;letter-spacing:.8px;">TIP</span>
                &nbsp; Use Draft to test the animation, then Final for your export.
                Files are saved in the renders folder.
            </div>
            """)

    render_button.click(
        fn=render_video,
        inputs=[code_input, ratio_input, quality_input],
        outputs=[video_output, file_output, status_output]
    )


app.queue().launch(
    share=True,
    debug=True,
    theme=gr.themes.Base(
        primary_hue="blue",
        secondary_hue="cyan",
        neutral_hue="slate"
    ),
    css=APP_CSS
)


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://572163eacf8b7cbfc5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://572163eacf8b7cbfc5.gradio.live
